In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")

from src.data import load_interactions, build_matrix
from src.evaluation import leave_k_out, evaluate
from src.models import RandomRecommender, PopularityRecommender, ALSRecommender

interactions = load_interactions()
matrix, user_to_row, song_to_col = build_matrix(interactions)
train, test, eval_users = leave_k_out(matrix)

print("matrix:", matrix.shape, "| filled:", matrix.nnz)
print("train:", train.nnz, "| eval users:", len(eval_users))

c:\Users\ayaan\OneDrive\Desktop\Spotify_Recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


matrix: (14857, 21219) | filled: 1209024
train: 1148764 | eval users: 12052


In [3]:
import pandas as pd

models = {
    "Random": RandomRecommender(),
    "Most popular": PopularityRecommender(),
    "ALS": ALSRecommender(),
}

results = {}
for name, model in models.items():
    model.fit(train)
    results[name] = evaluate(model.recommend, test, eval_users)

pd.DataFrame(results).T

c:\Users\ayaan\OneDrive\Desktop\Spotify_Recommender\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [00:05<00:00,  3.77it/s]


,precision@10,recall@10,ndcg@10
Random,0.0003,0.0005,0.0004
Most popular,0.0112,0.0225,0.0205
ALS,0.0636,0.1271,0.1135


In [4]:
from src.data import load_track_features
from src.models import ContentBasedRecommender

features = load_track_features(song_to_col)
print("songs with features:", len(features), "of", matrix.shape[1])

songs with features: 21219 of 21219


In [5]:
content = ContentBasedRecommender().fit(train, features)
results["Content-based"] = evaluate(content.recommend, test, eval_users)

pd.DataFrame(results).T

,precision@10,recall@10,ndcg@10
Random,0.0003,0.0005,0.0004
Most popular,0.0112,0.0225,0.0205
ALS,0.0636,0.1271,0.1135
Content-based,0.0065,0.0130,0.0108


In [6]:
u = eval_users[0]
col_to_song = {v: k for k, v in song_to_col.items()}

print("LISTENS TO:")
for c in list(content.seen(u))[:8]:
    print("  ", col_to_song[c])

print("\nCONTENT-BASED SUGGESTS:")
for c in content.recommend(u, 10):
    print("  ", col_to_song[c])

LISTENS TO:
   all time low|dear maria count me in
   all time low|merry christmas kiss my ass
   becky g|shower
   blink 182|after midnight
   blink 182|first date
   blink 182|ghost on the dance floor
   blink 182|happy holidays you bastard
   blink 182|i miss you

CONTENT-BASED SUGGESTS:
   the psychedelic furs|love my way
   rybičky 48|bohém zapomenutý dítě
   bulldog|más y más
   the smiths|heaven knows im miserable now
   simple plan|jet lag
   el bordo|silbando una ilusión
   karamelo santo|fruta amarga
   simple plan|addicted
   echo the bunnymen|lips like sugar
   the cab|moon


In [10]:
from src.models import ClusteredContentRecommender

clustered = ClusteredContentRecommender(n_clusters=3).fit(train, features)
results["Content-based (clustered)"] = evaluate(clustered.recommend, test, eval_users)

pd.DataFrame(results).T

,precision@10,recall@10,ndcg@10
Random,0.0003,0.0005,0.0004
Most popular,0.0112,0.0225,0.0205
ALS,0.0636,0.1271,0.1135
Content-based,0.0065,0.0130,0.0108
Content-based (clustered),0.0050,0.0100,0.0080


In [11]:
from src.models import HybridRecommender

als = models["ALS"]
hybrid = HybridRecommender(als, content, alpha=0.8).fit(train)
results["Hybrid (alpha=0.8)"] = evaluate(hybrid.recommend, test, eval_users)

pd.DataFrame(results).T

,precision@10,recall@10,ndcg@10
Random,0.0003,0.0005,0.0004
Most popular,0.0112,0.0225,0.0205
ALS,0.0636,0.1271,0.1135
Content-based,0.0065,0.0130,0.0108
Content-based (clustered),0.0050,0.0100,0.0080
Hybrid (alpha=0.8),0.0628,0.1257,0.1128


In [12]:
sweep = {}
for a in [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0]:
    h = HybridRecommender(als, content, alpha=a).fit(train)
    sweep[a] = evaluate(h.recommend, test, eval_users)

sweep_df = pd.DataFrame(sweep).T
sweep_df.index.name = "alpha (weight on ALS)"
sweep_df

,precision@10,recall@10,ndcg@10
alpha (weight on ALS),,,
0.0,0.0065,0.0130,0.0108
0.2,0.0317,0.0634,0.0594
0.4,0.0471,0.0942,0.0861
0.6,0.0568,0.1136,0.1027
0.8,0.0628,0.1257,0.1128
0.9,0.0636,0.1272,0.1149
1.0,0.0636,0.1271,0.1135


In [13]:
full = {}
for a in [0.9, 1.0]:
    h = HybridRecommender(als, content, alpha=a).fit(train)
    full[a] = evaluate(h.recommend, test, eval_users, sample=len(eval_users))

pd.DataFrame(full).T

,precision@10,recall@10,ndcg@10
0.9,0.0647,0.1294,0.1166
1.0,0.0644,0.1288,0.1149
